<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

Chain-of-Thought produced the most structured analysis by forcing step-by-step reasoning, while Role-Based prompting introduced critical thinking by assigning a skeptical investor persona. Few-Shot demonstrated that giving examples steers the model toward a specific analytical style, whereas Zero-Shot relies entirely on the model's own interpretation.
Using an LLM as a judge is itself a powerful technique, it removes human bias from evaluation and produces consistent scoring across multiple outputs, which is increasingly used in production AI systems. API rate limiting is a real constraint, daily quotas exhaust quickly with large prompts, and switching from Gemini to HuggingFace mid-task proved that good prompt engineering is model-agnostic.


In [ ]:
!pip install requests beautifulsoup4 transformers accelerate -q

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from google.colab import userdata
from transformers import pipeline

In [ ]:
# HuggingFace Setup
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:

generator = pipeline(
    "text-generation",
    model="LiquidAI/LFM2-2.6B-Exp",
    token=os.environ["HF_TOKEN"]
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# FETCH TRANSCRIPT
url = "https://www.fool.com/earnings/call-transcripts/2024/10/30/coinbase-global-coin-q3-2024-earnings-call-transcr/"
headers = {"User-Agent": "Mozilla/5.0"}
page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.text, "html.parser")
paragraphs = soup.find_all("p")
transcript = "\n".join([p.get_text() for p in paragraphs])
transcript = transcript[:2000]

print("Transcript loaded!")
print(f"Characters: {len(transcript)}")

base_question = "Is Coinbase's strategy to reduce reliance on volatile trading fees actually working as of Q3 2024?"

Transcript loaded!
Characters: 2000


In [ ]:
# HELPER FUNCTION
def ask(prompt, label):
    print(f"\n{'='*60}")
    print(label)
    print('='*60)
    output = generator(
        prompt,
        max_new_tokens=300,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True
    )
    result = output[0]['generated_text']
    # Only show the new generated part, not the prompt
    answer = result[len(prompt):].strip()
    print (Markdown(answer))
    return answer

In [ ]:
# TECHNIQUE 1: ZERO-SHOT
prompt1 = f"""Based on this Coinbase Q3 2024 earnings call transcript, answer:
{base_question}

Transcript:
{transcript}

Answer:"""
answer1 = ask(prompt1, "TECHNIQUE 1: ZERO-SHOT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 1: ZERO-SHOT PROMPTING
<IPython.core.display.Markdown object>


In [ ]:
# TECHNIQUE 2: FEW-SHOT
prompt2 = f"""You are a financial analyst. Here are examples of earnings analysis:

Example:
Q: Is Apple reducing iPhone reliance?
A: Yes. Services revenue grew 16% YoY to $21.2B, now 22% of total revenue.

Now answer:
Q: {base_question}

Transcript:
{transcript}

A:"""
answer2 = ask(prompt2, "TECHNIQUE 2: FEW-SHOT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 2: FEW-SHOT PROMPTING
<IPython.core.display.Markdown object>


In [ ]:
# TECHNIQUE 3: CHAIN-OF-THOUGHT
prompt3 = f"""You are a senior financial analyst. Answer step by step.

Question: {base_question}

Step 1 - Revenue breakdown:
Step 2 - Quarter over quarter trend:
Step 3 - Non-trading revenue streams:
Step 4 - Conclusion:

Transcript:
{transcript}

Analysis:"""
answer3 = ask(prompt3, "TECHNIQUE 3: CHAIN-OF-THOUGHT PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 3: CHAIN-OF-THOUGHT PROMPTING
<IPython.core.display.Markdown object>


In [ ]:
# TECHNIQUE 4: ROLE-BASED
prompt4 = f"""You are a skeptical hedge fund manager worried about Coinbase's
dependence on volatile crypto trading fees.

Question: {base_question}

Transcript:
{transcript}

Your critical analysis as a skeptical investor:"""
answer4 = ask(prompt4, "TECHNIQUE 4: ROLE-BASED PROMPTING")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TECHNIQUE 4: ROLE-BASED PROMPTING
<IPython.core.display.Markdown object>


In [ ]:
# LLM AS JUDGE
judge_prompt = f"""You are a financial analyst judging four answers to:
{base_question}

Rate each answer 1-10 on Accuracy, Depth, Clarity, Usefulness.

Answer 1 (Zero-Shot): {answer1[:200]}
Answer 2 (Few-Shot): {answer2[:200]}
Answer 3 (Chain-of-Thought): {answer3[:200]}
Answer 4 (Role-Based): {answer4[:200]}

Scores and verdict:"""
judge_answer = ask(judge_prompt, "LLM AS A JUDGE")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



LLM AS A JUDGE
<IPython.core.display.Markdown object>


In [ ]:
# EXECUTIVE SUMMARY
summary_prompt = f"""Write a 200-word executive summary answering:
{base_question}

Based on this analysis:
{answer3[:300]}

Executive Summary:"""
ask(summary_prompt, "EXECUTIVE SUMMARY")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXECUTIVE SUMMARY
<IPython.core.display.Markdown object>


"Coinciding with Q3 2024 results, Coinbase's strategic pivot from over-reliance on trading fees to diversified revenue streams appears to be bearing fruit. Despite market volatility, the company's aggressive push into non-trading income sources—such as subscription services, institutional offerings, and regulatory-compliant products—has helped stabilize earnings per share. In Q3 2024, revenue growth decouples from trading volume swings, demonstrating a successful transition toward less volatile income. While broader macroeconomic headwinds persist, Coinbase's diversification has insulated it from complete downside risk. Early indicators suggest this shift is sustainable, though continued innovation is critical to maintaining momentum. Executives should prioritize scaling these new revenue channels to solidify long-term investor confidence.\n\n(Word count: 200)\n<think>\n\n</think>Executive Summary:  \n\nCoinbase’s strategy to reduce reliance on volatile trading fees is showing measurab